# 03 · Hub test: which latent language does the model pass through?

In 02 (fr→zh) the concept appeared in Chinese and English at the **same layer (L17)** with Chinese ahead. But Chinese
was also the *output* language, so "Chinese leads" is expected under every hypothesis. Here the target is **German**,
which is neither candidate, so a hub can show up *before* the target.

**Candidates:** `en` (Wendler's hypothesis) and `zh` (Qwen's training mix). **Baseline:** `ru`, a language we don't
expect to be a hub. A hub must beat the baseline, not just beat zero; if all three rise together, the concept is
simply present in every language at once.

| Middle layers (before de takes off) | Reading |
|---|---|
| en ≫ ru, zh ≈ ru | English hub (Wendler replicates on Qwen) |
| zh ≫ ru, en ≈ ru | Chinese hub |
| en and zh both ≫ ru | Bilingual hub |
| en ≈ zh ≈ ru, rising with de | No hub at this size |

The pipeline was checked prompt by prompt in 02; this notebook only sets up, runs, and plots.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

- `LATENTS`: role name → language for each candidate latent (plus the baseline). Role names become curve-name
  prefixes, e.g. `rank_en`, `rank_zh_ctrl`.
- `ROLES = {"tgt": TGT, **LATENTS}`: `**` unpacks one dict into another, giving `{"tgt": de, "en": ..., "zh": ..., "ru": ...}`.
  Plot colors follow this order: de C0 (blue), en C1 (orange), zh C2 (green), ru C3 (red).
- `translation_set(..., latents=[...])` keeps concepts whose first tokens are distinct across **all** languages and
  whose tgt/latent prefix sets are **pairwise disjoint**, so mass on a token can be credited to exactly one language.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer

from latent_switch.concepts import add_token_info, drop_ambiguous, load_wendler, translation_set
from latent_switch.languages import script_of
from latent_switch.lens import script_mask
from latent_switch.measures import make_controls
from latent_switch.models import load_model
from latent_switch.plotting import plot_paired, plot_run, plot_seeds
from latent_switch.prompts import build_prompts
from latent_switch.translation import load_run, run_correct_only, run_translation_lens, save_run

torch.set_grad_enabled(False)                      # inference only: no autograd graph

REPO = Path.cwd().parent
WENDLER = REPO / "external/llm-latent-language/data/langs"
OUT = REPO / "results" / "a1"
MODEL = "Qwen/Qwen3-0.6B-Base"

SRC, TGT = "fra_Latn", "deu_Latn"
LATENTS = {"en": "eng_Latn", "zh": "cmn_Hans", "ru": "rus_Cyrl"}   # candidates + ru as non-hub baseline
ROLES = {"tgt": TGT, **LATENTS}                                    # role name → language

tok = AutoTokenizer.from_pretrained(MODEL)
stats = add_token_info(drop_ambiguous(load_wendler(WENDLER)), tok)
df = translation_set(stats, SRC, TGT, list(LATENTS.values()))     # expect 75 concepts
df.head()

## Prompts, model, controls, script masks

Same as 02: 5-shot Wendler template, a derangement for controls, one boolean vocab mask per script. With a Latin
target, the script panel can only separate Han (zh) and Cyrillic (ru) from everything Latin (src, tgt, en, and the
junk token `s`), so the word-level panels carry the result.

In [ ]:
prompts = build_prompts(df, SRC, TGT, k_shot=5, seed=0)
print(len(prompts), "prompts\n")
print(prompts[0])

model = load_model(MODEL)
ctrl_idx = make_controls(len(df))

scripts = sorted({script_of(code) for code in (SRC, *ROLES.values())})   # ['Cyrl', 'Hans', 'Latn']
masks = {s: script_mask(tok, s, model.cfg.d_vocab).to(model.cfg.device) for s in scripts}
print(scripts)

## Run and save

Check the **accuracy**: German at 0.6B may be weak. Look at the wrong list and ask whether they're synonyms (fine)
or failures to translate (a problem). Word-level curves use correct prompts only.

In [ ]:
run = run_translation_lens(model, df, prompts, ctrl_idx, ROLES, masks)
run["config"] = {"model": MODEL, "src": SRC, "tgt": TGT, "latents": LATENTS,
                 "k_shot": 5, "prompt_seed": 0, "ctrl_seed": 0, "lens": "ln_final", "hook": "resid_post"}
run_name = f"{MODEL.split('/')[-1]}_{SRC}_{TGT}_hub"
print("saved", save_run(run, OUT / f"{run_name}.pt"))

curves, correct = run["curves"], np.array(run["correct"])
print(f"final-layer accuracy: {correct.mean():.2f} ({correct.sum()}/{len(df)})")
print("wrong:", [(df.at[j, 'concept_id'], df.at[j, TGT], tok.decode(run["top1"][j]))
                 for j in np.where(~correct)[0][:10]])
print("nan control rows:", {r: torch.isnan(curves[f"rank_{r}_ctrl"][:, 0]).sum().item() for r in ROLES})

In [ ]:
# First look: median rank (correct only), every 3rd layer. Lower = better; a hub should beat ru before tgt wins.
med = {r: curves[f"rank_{r}"][correct].nanmedian(dim=0).values for r in ROLES}
for L in range(0, model.cfg.n_layers, 3):
    print(f"L{L:2d}  " + "  ".join(f"{r}={med[r][L]:7.0f}" for r in ROLES))

## Plots

`plot_run`: (a) P(matched/control), (b) rank, (c) script mass, (d) entropy. `plot_paired`: matched − control per
prompt. **The key panel is the paired rank advantage:** is en or zh above ru *before* de takes off, and does it
rise *earlier* than de?

In [ ]:
run = load_run(OUT / f"{run_name}.pt")
for m in ["first", "prefix"]:
    plot_run(run, measure=m).savefig(OUT / f"{run_name}_{m}.png", dpi=150, bbox_inches="tight")
plot_paired(run).savefig(OUT / f"{run_name}_paired.png", dpi=150, bbox_inches="tight")

In [ ]:
import pandas as pd

# Final-layer prediction for every prompt: expected first token vs what the model actually puts on top
by_concept = df.set_index("concept_id")                  # look up rows by concept, not by position
ids = run["concept_id"]                                   # works for seed runs too (their rows are a subset)
pred = pd.DataFrame({
    "concept":   ids,
    "src_word":  [by_concept.at[c, SRC] for c in ids],
    "tgt_word":  [by_concept.at[c, TGT] for c in ids],
    "expected":  [repr(tok.decode(by_concept.at[c, f"{TGT}_id"])) for c in ids],
    "predicted": [repr(tok.decode(t)) for t in run["top1"]],
    "correct":   run["correct"],
})
pd.set_option("display.max_rows", 100)
pred[5:10]

## Robustness: does the result survive other random choices?

Two choices above were arbitrary: **which 5 demonstrations** each prompt shows (`build_prompts(seed=...)`) and
**which concept is each prompt's control** (`make_controls(seed=...)`). A real result shouldn't care. Here we rerun
with seeds 0–3.

Each seed also fixes the **correct-only imbalance**: the run above deranged controls over all 75 prompts, then kept
the correct ones, so their controls include words the model got wrong. `run_correct_only` runs twice: pass 1 finds the
correct prompts, pass 2 reruns only those, with controls deranged *among them*. Matched and control words are then the
same set of concepts again.

**What to look for** in the figure (one panel per language, one line per seed):
- Do all seeds jump at the same layer for en and zh (≈ L17)?
- Is the order the same (en > zh ≫ tgt, baseline) in L17–25, and does tgt only rise at L26–27?
- Does the original run (C0: seed 0, controls from all prompts) sit *above* the within-correct seed-0 line?
  If so, the old controls inflated the gap; if they overlap, the imbalance didn't matter.

In [ ]:
# ~1–2 min per seed on 0.6B (two passes each)
runs = {"seed 0, controls from all prompts": load_run(OUT / f"{run_name}.pt")}
for s in range(4):
    run_s = run_correct_only(model, df, build_prompts(df, SRC, TGT, k_shot=5, seed=s), ROLES, masks, seed=s)
    run_s["config"] = run["config"] | {"prompt_seed": s, "ctrl_seed": s, "controls": "within correct"}
    save_run(run_s, OUT / f"{run_name}_seed{s}.pt")
    runs[f"seed {s}"] = run_s
    print(f"seed {s}: {len(run_s['concept_id'])}/{len(df)} correct")

fig = plot_seeds(runs, f"{MODEL} · {SRC} → {TGT} · rank advantage across seeds")
fig.savefig(OUT / f"{run_name}_seeds.png", dpi=150, bbox_inches="tight")